# 03 — SD3.5 Inpaint Test: Baseline vs LoRA

Raw SD3.5 inpaint only (no hard-restore / refinement / harmonization). The ONLY
difference between B0 and B1 is the trigger token in the prompt. Logic in
`LoRA/inference/`.

## 00. Clone + config + freeze check

In [ ]:
import subprocess, sys, json
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
sys.path.insert(0, str(REPO))
!pip install -q 'diffusers==0.35.2' 'transformers==4.46.3' 'huggingface_hub==0.36.0' 'accelerate==1.11.0' 'safetensors>=0.4.3' 'pillow>=10' 'datasets>=2.20' numpy
from LoRA.data.config import load_inpaint_eval_config, load_prompt_config
EVAL_CFG = load_inpaint_eval_config(REPO/'LoRA'/'configs'/'inpaint_eval.yaml')
PROMPTS  = load_prompt_config(REPO/'LoRA'/'configs'/'prompt_templates.yaml')
print(EVAL_CFG)

## 00b. Build the PIPE real golden eval set (run once)

PIPE pairs are real before/after photos. `source_img` (object erased) is the
inpaint input, `target_img` (real photo) is the ground truth, and the mask is
derived from `|target - source|`. Skip if the eval set already exists.

In [ ]:
from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
WORK = Path('/kaggle/working/vin_lora')
EVAL_RELEASE = Path(EVAL_CFG['evaluation_release'])
pe = EVAL_CFG.get('pipe_eval', {})
if not (EVAL_RELEASE/'cases.jsonl').exists():
    run_build_pipe_eval(WORK, eval_set=EVAL_RELEASE.name,
                        split=pe.get('split','test'),
                        person_only=pe.get('person_only', True),
                        limit=pe.get('limit'),
                        thresh=pe.get('mask_diff_threshold', 25),
                        dilate_px=pe.get('mask_dilate_px', 6))
else:
    print('PIPE eval set already exists ->', EVAL_RELEASE)

## 01-02. Load base model + verify it matches training provenance

In [ ]:
from LoRA.inference.sd35_inpaint_runner import SD35InpaintRunner, load_cases, render_inpaint_prompt
# Optional provenance match guard
PROV = Path('/kaggle/input/vinped-sd35m-lora-v1/training_provenance.json')
if EVAL_CFG.get('require_base_model_match') and PROV.exists():
    prov = json.load(open(PROV))
    assert prov['base_model_id'] == EVAL_CFG['base_model_id'], 'base model mismatch vs training'
runner = SD35InpaintRunner(EVAL_CFG['base_model_id']).load()
print('base model loaded')

## 03. Load frozen cases

In [ ]:
EVAL_RELEASE = Path(EVAL_CFG['evaluation_release'])
cases = load_cases(EVAL_RELEASE)
print(len(cases), 'cases')

## 04-06. Run B0 (baseline) and B1 (LoRA) — identical inputs except trigger token

In [ ]:
from LoRA.inference.inpaint_metrics import compute_case_metrics
RUN = Path('/kaggle/working/vin_lora/runs/inpaint_eval_v1_run_001')
(RUN/'baseline'/'images').mkdir(parents=True, exist_ok=True)
(RUN/'lora'/'images').mkdir(parents=True, exist_ok=True)

def run_condition(label, trigger, attach_lora):
    rows = []
    if attach_lora:
        runner.attach_lora(EVAL_CFG['adapter_path'], EVAL_CFG['adapter_name'], EVAL_CFG['adapter_weight'])
    else:
        runner.detach_lora()
    for case in cases:
        prompt = render_inpaint_prompt(PROMPTS.inpaint_prompt_template, case['prompt_fields'], trigger)
        for seed in EVAL_CFG['seed_list']:
            out = runner.run_case(case, EVAL_RELEASE, prompt, PROMPTS.negative_prompt, seed, EVAL_CFG)
            img_path = RUN/label/'images'/f"{case['case_id']}_s{seed}.png"
            out['image'].save(img_path)
            m = compute_case_metrics(EVAL_RELEASE/case['reference_path'], img_path,
                                     EVAL_RELEASE/case['mask_path'], case['expected_bbox_xyxy'],
                                     detector=None, dilate_px=EVAL_CFG.get('outside_mask_dilate_px',8))
            m.update({'case_id': case['case_id'], 'seed': seed,
                      'runtime_seconds': out['runtime_seconds'], 'cuda_peak_mb': out['cuda_peak_mb']})
            rows.append(m)
    return rows

baseline_rows = run_condition('baseline', '', attach_lora=False)
lora_rows     = run_condition('lora', PROMPTS.trigger_token, attach_lora=True)
print('baseline', len(baseline_rows), 'lora', len(lora_rows))

## 07-08. Paired comparison report

In [ ]:
from LoRA.inference.report import build_paired_comparison, print_summary, write_per_case_csv
write_per_case_csv(baseline_rows + lora_rows, RUN/'metrics_per_case.csv')
summary = build_paired_comparison(baseline_rows, lora_rows, RUN)
print_summary(summary)
print('\noutputs ->', RUN)